# Notebook para calcular los umbrales de seguridad

In [1]:
import numpy as np
import pandas as pd

Aplica a las lecturas meteorológicas los umbrales de seguridad de generación eólica del Parque Vader Piet. Los parámetros de la Vestas V90-3.0 MW proceden de `docs/ProductbrochureV90_3_0_UK.pdf`; la configuración del parque (10 turbinas) procede de `ingestion_consumo.ipynb`.

Definimos el parque y las condiciones de operación publicadas para la V90: arranque a 4 m/s, potencia nominal a 15 m/s y parada de protección (*cut-out*) a 25 m/s. Para este modelo se asume que `wind_speed_kmh` de la estación es representativa de la velocidad de referencia de la turbina, por lo que se compara directamente sin conversión de altura.

In [2]:
N_TURBINES = 10
TURBINE_RATED_POWER_KW = 3000.0
WIND_CAPACITY_KW = N_TURBINES * TURBINE_RATED_POWER_KW
CUT_IN_MS = 4.0
RATED_WIND_MS = 15.0
CUT_OUT_MS = 25.0

### Umbral de *cut-out* publicado

La tabla de datos operacionales de `ProductbrochureV90_3_0_UK.pdf` especifica `Cut-out wind speed: 25 m/s`. La conversión de unidades es directa:

$$25\,\mathrm{m/s}\times3.6=90\,\mathrm{km/h}$$

Por ello, `wind_alert` se marca como `CRITICAL` desde **90 km/h**. El folleto lista alturas de buje de 80 m y 105 m, pero no indica una altura de medida para el límite. Para este modelo se asume que el viento de estación representa la velocidad de referencia de la turbina, así que el notebook no incorpora ninguna extrapolación entre alturas.

Leemos el csv de lecturas y hacemos una copia del DataFrame para no modificar el original. Sobre `df` se van añadiendo las columnas de alerta en las celdas siguientes.

In [3]:
df_readings = pd.read_csv('../data/processed/2_data_validation/aruba_reading_filtered_by_station_distance.csv')
df = df_readings.copy()

La curva de potencia del folleto permite traducir los umbrales de potencia a viento. El 15% y el 40% del parque son 4,500 y 12,000 kW, respectivamente. Se interpolan en la curva de una V90 y se convierten únicamente de m/s a km/h, asumiendo que la lectura meteorológica representa la velocidad de referencia de la turbina.

In [4]:
threshold_40_pct = WIND_CAPACITY_KW * 0.40
threshold_15_pct = WIND_CAPACITY_KW * 0.15

# Curva V90-3.0 MW digitalizada del gráfico del folleto, a densidad estándar.
V90_WIND_MS = np.array([4.0, 4.5, 5.0, 5.5, 6.0, 6.5, 7.0, 7.5, 8.0, 8.5, 9.0, 9.5, 10.0,
                        10.5, 11.0, 11.5, 12.0, 12.5, 13.0, 13.5, 14.0, 14.5, 15.0])
V90_POWER_KW = np.array([97, 160, 233, 317, 414, 516, 641, 780, 944, 1115, 1296, 1478,
                         1668, 1862, 2061, 2254, 2447, 2606, 2732, 2843, 2928, 2984, 3000])

critical_wind_ms = np.interp(threshold_15_pct / N_TURBINES, V90_POWER_KW, V90_WIND_MS)
moderate_wind_ms = np.interp(threshold_40_pct / N_TURBINES, V90_POWER_KW, V90_WIND_MS)
WIND_CRITICAL_KMH = critical_wind_ms * 3.6
WIND_MODERATE_KMH = moderate_wind_ms * 3.6
WIND_CUT_OUT_KMH = CUT_OUT_MS * 3.6

print(f'15% de capacidad: {threshold_15_pct:,.0f} kW -> {critical_wind_ms:.2f} m/s -> {WIND_CRITICAL_KMH:.1f} km/h')
print(f'40% de capacidad: {threshold_40_pct:,.0f} kW -> {moderate_wind_ms:.2f} m/s -> {WIND_MODERATE_KMH:.1f} km/h')
print(f'Cut-out V90: {CUT_OUT_MS:.0f} m/s -> {WIND_CUT_OUT_KMH:.1f} km/h')

15% de capacidad: 4,500 kW -> 6.18 m/s -> 22.2 km/h
40% de capacidad: 12,000 kW -> 8.73 m/s -> 31.4 km/h
Cut-out V90: 25 m/s -> 90.0 km/h


## **Alerta de Déficit Eólico (Caída de Viento)** — Parque Vader Piet, 30 MW nominales. Calculamos las condiciones de déficit:
- Si existe `wind_power_kw` (potencia real):
  - Moderado: < 40% de capacidad nominal (12,000 kW) → respaldo solar / BESS
  - Crítico: < 15% de capacidad nominal (4,500 kW) → despacho térmico 100%
- Si no existe, se clasifica por `wind_speed_kmh`, usando directamente la curva V90:
  - Moderado: < 31.4 km/h (menos del 40% de la capacidad)
  - Crítico: < 22.2 km/h (menos del 15% de la capacidad)

In [5]:
if 'wind_power_kw' in df.columns:
    is_critical_deficit = df['wind_power_kw'] < threshold_15_pct
    is_moderate_deficit = df['wind_power_kw'] < threshold_40_pct
else:
    is_critical_deficit = df['wind_speed_kmh'] < WIND_CRITICAL_KMH
    is_moderate_deficit = df['wind_speed_kmh'] < WIND_MODERATE_KMH

Completamos la alerta eólica con la condición de viento extremo y asignamos la etiqueta `wind_alert`:
- Crítico por *cut-out*: >= 90 km/h, equivalente a los 25 m/s publicados para la V90 (las turbinas frenan por protección)
- Etiquetas posibles: `CRITICAL` → `MODERATE` → `NORMAL`. Tanto el déficit crítico como el *cut-out* se etiquetan como `CRITICAL`.

**Cómo se asigna la etiqueta (`.loc`)**: las condiciones (`is_cut_out`, `is_critical_deficit`, `is_moderate_deficit`) son Series de `True`/`False`, una por fila. Primero se marca toda la columna como `NORMAL` y luego `df.loc[condición, 'wind_alert'] = etiqueta` sobrescribe solo las filas donde la condición es `True`, todas a la vez y sin recorrer el DataFrame con un bucle.

Una misma lectura puede cumplir varias condiciones a la vez (por ejemplo, 10 km/h está por debajo de 25 y de 15 km/h). Como cada asignación pisa a la anterior, **se escriben de menos a más grave**: así la última, la más grave, es la que se queda. Si se invirtiera el orden, una lectura crítica acabaría marcada como moderada.

In [6]:
if 'wind_speed_kmh' in df.columns:
    is_cut_out = df['wind_speed_kmh'] >= WIND_CUT_OUT_KMH
else:
    is_cut_out = pd.Series(False, index=df.index)

Asigna una etiqueta de alerta eólica: se parte de 'NORMAL' y cada .loc sobrescribe las filas que cumplen su condición. Gana la última asignación, por eso el orden va de menos a más grave

In [7]:
df['wind_alert'] = 'NORMAL'
df.loc[is_moderate_deficit, 'wind_alert'] = 'MODERATE'
df.loc[is_critical_deficit, 'wind_alert'] = 'CRITICAL'
df.loc[is_cut_out, 'wind_alert'] = 'CRITICAL'

Mostramos cuántas lecturas hay en cada nivel de alerta

In [8]:
print(df['wind_alert'].value_counts())

wind_alert
MODERATE    205746
CRITICAL    102005
NORMAL        9265
Name: count, dtype: int64


## Exportación de la tabla enriquecida

Guardamos una nueva tabla sin modificar el CSV de entrada. El fichero incluye todas las columnas originales y el nuevo campo `wind_alert`, para que los siguientes pasos del pipeline puedan utilizar los umbrales calculados.

In [9]:
CSV_OUTPUT_PATH = '../data/processed/3_security_thresholds/aruba_readings_with_security_thresholds.csv'
JSON_OUTPUT_PATH = '../data/processed/3_security_thresholds/aruba_readings_with_security_thresholds.json'

df.to_csv(CSV_OUTPUT_PATH, index=False)
df.to_json(JSON_OUTPUT_PATH, orient='records', indent=2, force_ascii=False)

print(f'Tabla enriquecida exportada en: {CSV_OUTPUT_PATH}')
print(f'Tabla enriquecida exportada en: {JSON_OUTPUT_PATH}')

Tabla enriquecida exportada en: ../data/processed/3_security_thresholds/aruba_readings_with_security_thresholds.csv
Tabla enriquecida exportada en: ../data/processed/3_security_thresholds/aruba_readings_with_security_thresholds.json
